* usage log 확인 

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w4" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : d:\gangsa\hanwha-agent


In [ ]:
import json

from sqlalchemy import func, select

import app.integrations.factory as factory
from app.db.session import session_scope
from app.integrations.ports import LLMResult
from app.models import UsageLog
from app.services import chat_service

# 가짜 llm 
class StubLLM:
    name = "stub"

    def __init__(self, replies: list[str]) -> None:
        self.replies = list(replies)
        self.calls = 0

    def answer(self, *, question: str, contexts: list[dict], user: dict) -> LLMResult:
        text = self.replies[min(self.calls, len(self.replies) - 1)]
        self.calls += 1
        return LLMResult(
            text=text,
            model="claude-haiku-4-5",
            input_tok=1200,
            output_tok=300,
            cost_krw=3.8,    
            latency_ms=900,
        )

# 규약을 지키는 응답 샘플 
GOOD = json.dumps(
    {
        "answer": "부산 출장 숙박비는 1박 7만원까지 지원됩니다.",
        "sources": [
            {
                "doc_id": "DOC-HR-014",
                "title": "국내출장 여비 규정",
                "version": "v2.0",
                "locator": "제12조",
            }
        ],
        "enough_evidence": True,
    },
    ensure_ascii=False,
)

# sources가 빠진 잘못된 응답 샘플
BAD = json.dumps(
    {"answer": "부산 출장 숙박비는 1박 7만원까지 지원됩니다.", "enough_evidence": True},
    ensure_ascii=False,
)

# 실행 번호 run_id로 쌓인 usage_log 레코드 수 카운트 해주는 함수 
def rows_for(run_id: str) -> int:
    with session_scope() as session:
        return session.scalar(
            select(func.count()).select_from(UsageLog).where(UsageLog.run_id == run_id)
        )

print("재료 준비 완료 — StubLLM  GOOD  BAD  rows_for()")

재료 준비 완료 — StubLLM  GOOD  BAD  rows_for()


In [3]:
def ask_with(stub: StubLLM):
    original = factory.get_llm 
    factory.get_llm = lambda: stub 
    try:
        return chat_service.ask(question="부산 출장 숙박비 한도가 얼마인가요?")
    finally:
        factory.get_llm = original

once = ask_with(StubLLM([GOOD]))
print(f"한번에 성공: run_id={once.run_id}, attempts={once.attempts}, usage_logs 개수={rows_for(once.run_id)}개")

retried = ask_with(StubLLM([BAD, GOOD]))
print(f"재시도 경우: run_id={retried.run_id}, attempts={retried.attempts}, usage_logs 개수={rows_for(retried.run_id)}개")


한번에 성공: run_id=RUN-8822, attempts=1, usage_logs 개수=1개
11:22:33 WARNING  app.services.chat_service: 스키마 위반 1/3회 : sources: Field required
재시도 경우: run_id=RUN-8823, attempts=2, usage_logs 개수=2개


In [5]:
# 사용량 집계 
from sqlalchemy import func, select 

from app.db.session import session_scope
from app.models import Run, UsageLog, User

with session_scope() as session:
    per_person = session.execute(
        select(
            User.name, 
            User.emp_no, 
            func.count(UsageLog.id), 
            func.sum(UsageLog.cost_krw), 
        )
        .join(Run, UsageLog.run_id == Run.id)
        .join(User, Run.user_id == User.id)
        .group_by(User.name, User.emp_no)
    ).all() 

    print("***사람별 결과***")
    for name, emp_no, count, total in per_person:
        print(f" {name} ({emp_no}) : {count}건 {round(total or 0, 1)}원")

    in_tok, out_tok = session.execute(
        select(func.sum(UsageLog.input_tok), func.sum(UsageLog.output_tok))
    ).one()
    in_tok, out_tok = in_tok or 0, out_tok or 0 
    share = out_tok / (in_tok + out_tok) * 100 if (in_tok + out_tok) else 0.0

    print("***토큰 비중***")
    print(f" 입력 {in_tok}, 출력 {out_tok}")
    print(f" 출력 비중 : {share:.1f} %")


***사람별 결과***
 김민준 (2019-0412) : 3건 11.4원
***토큰 비중***
 입력 3600, 출력 900
 출력 비중 : 20.0 %
